In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import akita_utils_forplotting as utils
import numpy as np
import pandas as pd
import sys, os, psutil, time
import matplotlib.pyplot as plt
import pysam
    
from skimage.transform import resize


#from optparse import OptionParser


nt = ['A', 'T', 'C', 'G']

2024-11-20 13:10:52.022208: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /pollard/home/shzhang/miniconda3/envs/dchic1/lib/R/library
2024-11-20 13:10:52.022240: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2024-11-20 13:10:54.969800: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /pollard/home/shzhang/miniconda3/envs/dchic1/lib/R/library
2024-11-20 13:10:54.969832: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2024-11-20 13:10:54.969858: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not app

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 sequence (InputLayer)          [(None, 1048576, 4)  0           []                               
                                ]                                                                 
                                                                                                  
 stochastic_reverse_complement   ((None, 1048576, 4)  0          ['sequence[0][0]']               
 (StochasticReverseComplement)  , ())                                                             
                                                                                                  
 stochastic_shift (StochasticSh  (None, 1048576, 4)  0           ['stochastic_reverse_complement[0
 ift)                                                            ][0]']                     

                                                                                                  
 re_lu_8 (ReLU)                 (None, 4096, 96)     0           ['max_pooling1d_7[0][0]']        
                                                                                                  
 conv1d_8 (Conv1D)              (None, 4096, 96)     46080       ['re_lu_8[0][0]']                
                                                                                                  
 batch_normalization_8 (BatchNo  (None, 4096, 96)    384         ['conv1d_8[0][0]']               
 rmalization)                                                                                     
                                                                                                  
 max_pooling1d_8 (MaxPooling1D)  (None, 2048, 96)    0           ['batch_normalization_8[0][0]']  
                                                                                                  
 re_lu_9 (

 add_2 (Add)                    (None, 512, 96)      0           ['add_1[0][0]',                  
                                                                  'dropout_2[0][0]']              
                                                                                                  
 re_lu_17 (ReLU)                (None, 512, 96)      0           ['add_2[0][0]']                  
                                                                                                  
 conv1d_17 (Conv1D)             (None, 512, 48)      13824       ['re_lu_17[0][0]']               
                                                                                                  
 batch_normalization_17 (BatchN  (None, 512, 48)     192         ['conv1d_17[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 re_lu_18 

 batch_normalization_25 (BatchN  (None, 512, 48)     192         ['conv1d_25[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 re_lu_26 (ReLU)                (None, 512, 48)      0           ['batch_normalization_25[0][0]'] 
                                                                                                  
 conv1d_26 (Conv1D)             (None, 512, 96)      4608        ['re_lu_26[0][0]']               
                                                                                                  
 batch_normalization_26 (BatchN  (None, 512, 96)     384         ['conv1d_26[0][0]']              
 ormalization)                                                                                    
                                                                                                  
 dropout_7

                                                                                                  
 batch_normalization_32 (BatchN  (None, 512, 512, 48  192        ['conv2d_4[0][0]']               
 ormalization)                  )                                                                 
                                                                                                  
 dropout_9 (Dropout)            (None, 512, 512, 48  0           ['batch_normalization_32[0][0]'] 
                                )                                                                 
                                                                                                  
 add_9 (Add)                    (None, 512, 512, 48  0           ['symmetrize2d_1[0][0]',         
                                )                                 'dropout_9[0][0]']              
                                                                                                  
 symmetriz

 ormalization)                  )                                                                 
                                                                                                  
 dropout_12 (Dropout)           (None, 512, 512, 48  0           ['batch_normalization_38[0][0]'] 
                                )                                                                 
                                                                                                  
 add_12 (Add)                   (None, 512, 512, 48  0           ['symmetrize2d_4[0][0]',         
                                )                                 'dropout_12[0][0]']             
                                                                                                  
 symmetrize2d_5 (Symmetrize2D)  (None, 512, 512, 48  0           ['add_12[0][0]']                 
                                )                                                                 
          

In [3]:
import h5py
import pyBigWig
import intervaltree
import basenji
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# hic imports
import cooler
from cooltools.lib.numutils import observed_over_expected, adaptive_coarsegrain
from cooltools.lib.numutils import interpolate_bad_singletons, set_diag, interp_nan
from astropy.convolution import Gaussian2DKernel
from astropy.convolution import convolve

import collections 

In [4]:
import cooltools.lib.plotting
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import EngFormatter
import seaborn as sns
from pybedtools import BedTool
import math


In [5]:
#REMEMBER!!!!!!! +1MB SEQUENCE GOES IN, BUT MAKES PREDICTION FOR 917504BP
#1MB CORRESPONDS TO 512, 2048 BP BINS THAT GETS CUT DOWN TO 448 BINS 

In [6]:
def format_ticks(ax, x=True, y=True, rotate=True):
    if y:
        ax.yaxis.set_major_formatter(bp_formatter)
    if x:
        ax.xaxis.set_major_formatter(bp_formatter)
        ax.xaxis.tick_bottom()
    if rotate:
        ax.tick_params(axis='x',rotation=45)
        
def save_figures_to_pdf(output_pdf_path, figures):
    with PdfPages(output_pdf_path) as pdf:
        for fig in figures:
            pdf.savefig(fig)

In [7]:
def read_fasta(fa):
    #read in fasta and return sequence only 
    seq=''
    with open(fa, "r") as fa_file:
        for line in fa_file:
            if not (line.startswith('>')):
                seq += line.strip()
    fa_file.close()                
    return(seq)
    

In [8]:
#TOTAL AKITA INPUT LENGTH 

akita_seq_len=1048576


#HG38 REF FA
hg38_file = '/pollard/home/shzhang/akita/run_akita/refs/hg38.fa'
fasta_hg38 = pysam.Fastafile(hg38_file)

#GENCODE
gene_annot = pd.read_csv('/pollard/data/genetics/GENCODE/release32/gencode.v32.genes_only.bed', sep = '\t', names = ['CHROM', 'start_gene', 'end_gene', 'Gene'])
gene_annot[gene_annot.CHROM != 'chrM']

,CHROM,start_gene,end_gene,Gene
0,chr1,11869,14409,DDX11L1
1,chr1,14404,29570,WASH7P
2,chr1,17369,17436,MIR6859-1
3,chr1,29554,31109,MIR1302-2HG
4,chr1,30366,30503,MIR1302-2
...,...,...,...,...
60567,chrY,57184101,57197337,IL9R
60568,chrY,57190738,57208756,AJ271736.1
60569,chrY,57201143,57203357,WASIR1
60570,chrY,57207346,57212230,WASH6P


In [9]:
data_dir='/pollard/data/projects/shzhang/mdanderson/'
patient_data_dir=f'{data_dir}by_patient_20240529/'

In [10]:
meta=pd.read_csv(f'{data_dir}meta_sv2.csv')
meta['tumor_id']=meta['tumor'].str.split('-').str.get(0)
meta

,Patient,Tumor_Type,Timepoint,normal,tumor,gender,tumor_id
0,Patient1,DDLPS,baseline,226294-WG01,226295-WG01,XX,226295
1,Patient1,DDLPS,On_treatment,226294-WG01,226296-WG01,XX,226296
2,Patient1,DDLPS,Surgery,226294-WG01,226298-WG01,XX,226298
3,Patient1,DDLPS,Surgery,226294-WG01,226297-WG01,XX,226297
4,Patient2,DDLPS,baseline,226299-WG01,226300-WG01,XX,226300
...,...,...,...,...,...,...,...
67,Patient23,DDLPS,baseline,227190-WG01,227180-WG01,XX,227180
68,Patient23,DDLPS,On_treatment,227190-WG01,227181-WG01,XX,227181
69,Patient23,DDLPS,Surgery,227190-WG01,227182-WG01,XX,227182
70,Patient24,UPS,baseline,227191-WG01,227183-WG01,XX,227183


In [32]:
meta[meta['Patient']=='Patient6']

,Patient,Tumor_Type,Timepoint,normal,tumor,gender,tumor_id
18,Patient6,DDLPS,On_treatment,226317-WG01,226318-WG01,XX,226318
19,Patient6,DDLPS,Surgery,226317-WG01,226319-WG01,XX,226319
20,Patient6,DDLPS,Relapse,226317-WG01,226320-WG01,XX,226320


In [11]:
#read in COSMIC gene information 
cosmic=pd.read_csv('/pollard/home/shzhang/data/COSMIC_census_2023.csv')
cosmic['chr'] = 'chr' + cosmic['Genome Location'].str.split(':').str[0]
cosmic['coords'] = cosmic['Genome Location'].str.split(':').str[1]
cosmic['start'] = cosmic['coords'].str.split('-').str[0]
cosmic['end'] = cosmic['coords'].str.split('-').str[1]
cosmic

,Gene Symbol,Name,Entrez GeneId,Genome Location,Tier,Hallmark,Chr Band,Somatic,Germline,Tumour Types(Somatic),...,Role in Cancer,Mutation Types,Translocation Partner,Other Germline Mut,Other Syndrome,Synonyms,chr,coords,start,end
0,A1CF,APOBEC1 complementation factor,29974.0,10:50799421-50885675,2,NaN,11.23,yes,NaN,melanoma,...,oncogene,Mis,NaN,NaN,NaN,"29974,A1CF,ACF,ACF64,ACF65,APOBEC1CF,ASP,ENSG0...",chr10,50799421-50885675,50799421,50885675
1,ABI1,abl-interactor 1,10006.0,10:26746593-26860935,1,Yes,12.10,yes,NaN,AML,...,"TSG, fusion",T,KMT2A,NaN,NaN,"10006,ABI-1,ABI1,E3B1,ENSG00000136754.17,Q8IZP...",chr10,26746593-26860935,26746593,26860935
2,ABL1,v-abl Abelson murine leukemia viral oncogene h...,25.0,9:130713946-130885683,1,Yes,34.12,yes,NaN,"CML, ALL, T-ALL",...,"oncogene, fusion","T, Mis","BCR, ETV6, NUP214",NaN,NaN,"25,ABL,ABL1,ENSG00000097007.17,JTK7,P00519,c-A...",chr9,130713946-130885683,130713946,130885683
3,ABL2,"c-abl oncogene 2, non-receptor tyrosine kinase",27.0,1:179099327-179229601,1,NaN,25.20,yes,NaN,AML,...,"oncogene, fusion",T,ETV6,NaN,NaN,"27,ABL2,ABLL,ARG,ENSG00000143322.19,P42684",chr1,179099327-179229601,179099327,179229601
4,ACKR3,atypical chemokine receptor 3,57007.0,2:236569641-236582358,1,Yes,37.30,yes,NaN,lipoma,...,"oncogene, fusion",T,HMGA2,NaN,NaN,"57007,ACKR3,CMKOR1,CXCR7,ENSG00000144476.5,GPR...",chr2,236569641-236582358,236569641,236582358
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
731,ZNF429,zinc finger protein 429,353088.0,19:21505564-21538078,2,NaN,12.00,yes,NaN,GBM,...,NaN,Mis,NaN,NaN,NaN,"353088,ENSG00000197013.9,Q86V71,ZNF429",chr19,21505564-21538078,21505564,21538078
732,ZNF479,zinc finger protein 479,90827.0,7:57119614-57139864,2,NaN,11.20,yes,NaN,"lung cancer, bladder carcinoma, prostate carci...",...,NaN,Mis,NaN,NaN,NaN,"90827,ENSG00000185177.12,KR19,Q96JC4,ZNF479",chr7,57119614-57139864,57119614,57139864
733,ZNF521,zinc finger protein 521,25925.0,18:25061926-25352152,1,NaN,11.20,yes,NaN,ALL,...,"oncogene, fusion",T,PAX5,NaN,NaN,"25925,EHZF,ENSG00000198795.10,Evi3,Q96K83,ZNF521",chr18,25061926-25352152,25061926,25352152
734,ZNRF3,zinc and ring finger 3,84133.0,22:28883592-29057487,2,NaN,12.10,yes,NaN,"colorectal cancer, adrenocortical carcinoma, g...",...,TSG,"N, F, Mis",NaN,NaN,NaN,"84133,BK747E2.3,ENSG00000183579.15,FLJ22057,KI...",chr22,28883592-29057487,28883592,29057487


In [12]:
#np.unique(cosmic['Gene Symbol'])

In [13]:
#drop rows without coords
cosmic_subset = cosmic[(cosmic['start'].notna()) & (cosmic['start'] != '') & 
                 (cosmic['end'].notna()) & (cosmic['end'] != '')]

single_cols = ['chr', 'start', 'end', 'Gene Symbol']
cosmic_subset = cosmic_subset[single_cols]

cosmic_BED=BedTool.from_dataframe(cosmic_subset.astype({'start': int, 'end': int}))
#cosmic_BED=BedTool.from_dataframe(cosmic_subset)


In [14]:
cosmic_subset[cosmic_subset['Gene Symbol']=='TNFRSF14']

,chr,start,end,Gene Symbol
683,chr1,2556366,2563829,TNFRSF14


In [15]:
#ok, what we want to do is plot the changes at every breakpoint
#we also want to figure out which bin the gene is in, so we can make a note of that
#we also want to get the bin at which the boundaries change, so we can also make a note of that 

In [29]:
#write separate plotting function so this isn't too messy
#for plotting post akita predictions 

def plot_complex_pred(mat_list, label_list, bkpt_bin_list, gene_start_bin_list, gene_end_bin_list, gene_name_start_list, gene_name_end_list, gene_len):

    #order of lists is REF_left, REF_right, ALT
     
    fig=plt.figure(figsize=(7.5,6))
    target_index = 0
    vmin=-2; vmax=2

    plt.subplot(121) 
    im = plt.matshow(mat_list[1], fignum=False, cmap= 'RdBu_r', vmax=vmax, vmin=vmin)
    
    for g in gene_start_bin_list:
        plt.axvline(x = g, color = 'limegreen', linestyle = '--', linewidth=1, label = 'gene start')
    for g in gene_end_bin_list:
        plt.axvline(x = g, color = 'forestgreen', linestyle = '--', linewidth=1, label = 'gene end')
    for g in bkpt_bin_list:
        plt.axvline(x = g, color = 'blue', linestyle = 'dotted', linewidth=1, label = 'other bkpts ')
        
    # Add gene name annotations
    for gene_name, start_bin in zip(gene_name_start_list, gene_start_bin_list):
        plt.text(
            x=start_bin, y=1.2 * mat_list[1].shape[0],  # Position annotation near the top of the matrix
            s=gene_name,
            color='limegreen', fontsize=10, rotation=45, ha='right'
        )
    for gene_name, end_bin in zip(gene_name_end_list, gene_end_bin_list):
        plt.text(
            x=end_bin, y=1.2 * mat_list[1].shape[0],  # Position annotation near the top of the matrix
            s=gene_name,
            color='forestgreen', fontsize=10, rotation=45, ha='right'
        )
    
    
    plt.colorbar(im, fraction=.04, pad = 0.05, ticks=[-2,-1, 0, 1,2]);
    plt.title(label_list[0], y=1.15)
    
    plt.subplot(122) 
    im = plt.matshow(mat_list[0], fignum=False, cmap= 'RdBu_r', vmax=vmax, vmin=vmin)
    gene_len_n_bin=math.floor(gene_len/2048)
    plt.axvline(x = 224-(gene_len_n_bin/2), color = 'limegreen', linestyle = '--', linewidth=1, label = 'gene start')
    plt.axvline(x = 224+(gene_len_n_bin/2), color = 'forestgreen', linestyle = '--', linewidth=1, label = 'gene end')


    plt.colorbar(im, fraction=.04, pad = 0.05, ticks=[-2,-1, 0, 1,2]);
    plt.title(label_list[1], y=1.15)

    
    
    fig.suptitle(label_list[2], fontsize=16, y=0.90)

        
    plt.tight_layout()
    plt.subplots_adjust(top=0.75)
    return(fig)

In [30]:
#make akita predictions for each walk and plot 
def pred_akita_genecenter_COSMIC(fa_dir, fa_name, patient_num):
    bed_name=fa_name.replace('fa', 'bed')
    name_parts = bed_name.rsplit('_', 2)


    #read in corresponding fa sequence
    seq=read_fasta(f'{fa_dir}{fa_name}')
    
    try:

        #read in bed file and figure out where to center the file 
        bed=pd.read_csv(f'{patient_data_dir}walks_bed/' + bed_name, sep='\t', header=None)
        bed.columns = ['chr', 'start', 'end', 'strand', 'cn', '5']
        bed['chr'] = bed['chr'].astype('str') 
        bed['span']=abs(bed['end']-bed['start']) 
        bed['span_cumsum'] = bed['span'].cumsum().astype(int)
        bed['bed_idx']=bed.index
    
        print('bed file', bed)
        walks_BED=BedTool.from_dataframe(bed[['chr', 'start', 'end', 'strand', 'cn', 'span', 'span_cumsum', 'bed_idx']])

        
        #overlap walks bed with COSMIC gene list 
        genes_in_map_BED = walks_BED.intersect(cosmic_BED, wa = True, wb = True)


        
        if genes_in_map_BED == '':
            #only want to visualize if gene is in the fasta 
            pass

        else:
            genes_in_map = (genes_in_map_BED.to_dataframe()
                        .rename(columns = {'strand': 'span', 'itemRgb': 'gene_chrom', 'name': 'strand', 
                                           'score': 'cn', 'thickStart': 'span_cumsum', 'thickEnd': 'bed_idx',
                                           'blockCount':'gene_start', 'blockSizes': 'gene_end', 
                                           'blockStarts':'gene_name'}))
            genes_in_map=genes_in_map[['chrom', 'start', 'end', 'strand', 'cn', 'span', 'span_cumsum', 'bed_idx',
                                       'gene_chrom', 'gene_start', 'gene_end','gene_name']]
            
            #print(genes_in_map)
            
            
            
            figures=[]
            #loop through each gene, center it, plot 
            for gene in range(0, len(genes_in_map)):
            #for gene in range(0, 2):



                gene_intersect_row=genes_in_map.loc[gene]
                print('gene', gene_intersect_row.gene_name)
                #print('gene_row', gene_row)

 
                try:
        
        
                    #get row in original bed that corresponds to gene row
                    gene_row_idx=bed[((bed['chr']==gene_intersect_row['chrom']) 
                                      & (bed['start']==gene_intersect_row.start)
                                     & (bed['end']==gene_intersect_row['end']))].index.values[0] 
                
                    #get gene info 
                    gene_len=abs(gene_intersect_row.gene_end-gene_intersect_row.gene_start)
                    gene_mid_coord=np.floor(gene_intersect_row.gene_start + (gene_len/2))
                    
                   
                    
                    #get first, last rows, and subsets of bed before and after bkpt 
                    first_row=bed.iloc[0,:]
                    last_row=bed.iloc[(len(bed)-1), :]
                    gene_row=bed.iloc[gene_row_idx, :]
                    bed_to_gene=bed.iloc[0:(gene_row_idx),:]
                                      

                    #get relevant lengths
                    total_span=np.sum(bed['span'])
                    left_to_gene_mid_coord_span=gene_mid_coord-gene_row['start']
                    #gene_mid_coord_to_right=gene_row['end']-gene_mid_coord
                    left_to_gene_span=int(np.sum(bed_to_gene['span']) + left_to_gene_mid_coord_span)
                    gene_to_right_span=total_span-left_to_gene_span 

                    #print(left_to_gene_span)

                    # left sequence 
                    if left_to_gene_span>=utils.half_patch_size:
                        ALT_left=seq[(left_to_gene_span - utils.half_patch_size ):left_to_gene_span]
                        
                        #for gene annot
                        left_start=left_to_gene_span - utils.half_patch_size 

                    else:   
                        #need to pad left from reference 
                        diff_len=utils.half_patch_size-left_to_gene_span
                        #print('diff len', diff_len)

                        first_row_idx=first_row.start
                        ALT_fromref_left=fasta_hg38.fetch(first_row.chr, first_row_idx-diff_len, first_row_idx).upper()

                        ALT_left=ALT_fromref_left + seq[0:len_left_to_bkpt]
                        
                        #for gene annot
                        left_start=-diff_len


                    # right sequence 
                    if gene_to_right_span>utils.half_patch_size:
                        ALT_right=seq[left_to_gene_span: (left_to_gene_span + utils.half_patch_size)]

                    else:
                        #need to pad right from reference        
                        diff_len=utils.half_patch_size-gene_to_right_span

                        last_row_idx=last_row.end
                        ALT_fromref_right=fasta_hg38.fetch(last_row.chr, last_row_idx, last_row_idx + diff_len).upper()

                        ALT_right=seq[left_to_gene_span:total_span] + ALT_fromref_right


                    ALT_seq = ALT_left + ALT_right
                    
                    #print('left start', left_start)
                    
                    #########################
                    #finally, get the REF -- center at the gene 
                    left_ref_coords=[gene_intersect_row.gene_chrom, gene_mid_coord-utils.half_patch_size, 
                                                  gene_mid_coord+utils.half_patch_size]


                    REF_seq_gene=fasta_hg38.fetch(left_ref_coords[0], left_ref_coords[1], left_ref_coords[2]).upper()


                    ####################
                    
                    if REF_seq_gene==ALT_seq:
                        #want where breakpoints and genes differ 
                        pass
                    


                    else:
                        
                        #check sequence lengths, and then run akita 
                        assert len(REF_seq_gene) == utils.seq_length, 'REF_seq_left is not right length'
                        assert len(ALT_seq) == utils.seq_length, 'ALT_seq is not right length'



                        #this is where akita is actually run, output vector size is (99681,)
                        REF_gene_vec,  ALT_vec = utils.vector_from_seq(REF_seq_gene),  utils.vector_from_seq(ALT_seq)
                        REF_gene_mat,  ALT_mat = utils.mat_from_vector(REF_gene_vec), utils.mat_from_vector(ALT_vec)


                        #order is REF_left, REF_right, ALT
                        mat_list=[REF_gene_mat, ALT_mat]

                        
                        
                        #######################

                        #we want to get indices of the breakpoints, if others are in the window 


                        span_cumsum_list=np.array(bed['span_cumsum'].tolist())
                        bkpt_pos_from_left_all= span_cumsum_list-left_start


                        #subset to those less than akita window size
                        bkpt_pos_from_left=bkpt_pos_from_left_all[(bkpt_pos_from_left_all>0) & (bkpt_pos_from_left_all<utils.MB)]
                        #print(bkpt_pos_from_left)

                        #get the bin index
                        bkpt_pos_from_left=bkpt_pos_from_left.tolist()
                        bkpt_bin_list_all=[utils.get_bin(bkpt) for bkpt in bkpt_pos_from_left]
                        bkpt_bin_list = [bkpt for bkpt in bkpt_bin_list_all if 0 < bkpt <= 448]



                        ########################
                        #finally, get the positions of the gene(s)

                        gene_start_bin_list=[]
                        gene_end_bin_list=[]
                        gene_name_start_list=[]
                        gene_name_end_list=[]


                        #check the start of the gene 
                        for g in range(0, len(genes_in_map)):
                            gene_annot_row=genes_in_map.loc[g]
                            bed_g_idx=gene_annot_row['bed_idx']
                            bed_g_row=bed.iloc[bed_g_idx, :]
                            bed_to_gene=bed.iloc[0:(bed_g_idx),:]


                            #check that start coord is in the bed
                            if (gene_annot_row['gene_start']>bed_g_row.start & gene_annot_row['gene_start']<bed_g_row.end):
                                len_to_gene_start_row=gene_annot_row['gene_start'] - (bed_g_row.start) 
                                left_to_gene_start=np.sum(bed_to_gene['span']) + len_to_gene_start_row
                                gene_start_pos_window=utils.get_bin(left_to_gene_start - left_start)
                                if (gene_start_pos_window>0) & (gene_start_pos_window<448):
                                    gene_start_bin_list.append(gene_start_pos_window)
                                    gene_name_start_list.append(gene_annot_row['gene_name'])


                            #check that end coord is in bed
                            if (gene_annot_row['gene_end']>bed_g_row.start & gene_annot_row['gene_end']<bed_g_row.end):
                                len_to_gene_end_row=gene_annot_row['gene_end'] - (bed_g_row.start) 
                                left_to_gene_end=np.sum(bed_to_gene['span']) + len_to_gene_end_row
                                gene_end_pos_window=utils.get_bin(left_to_gene_end - left_start)
                            if (gene_end_pos_window>0) & (gene_end_pos_window<448):
                                gene_end_bin_list.append(gene_end_pos_window)
                                gene_name_end_list.append(gene_annot_row['gene_name'])


                        ####################

                        #get label list 
                        label_list=[f'Row {gene_row_idx}, centered at {gene_intersect_row.gene_name}', 
                                    'REF at {} \n {}:{}-{}'.format(gene_intersect_row.gene_name, left_ref_coords[0], left_ref_coords[1], left_ref_coords[2]), 
                                    f'{fa_name[:-3]}']

                        #print(label_list)

                        # plot!
                        fig=plot_complex_pred(mat_list, label_list, bkpt_bin_list, gene_start_bin_list, gene_end_bin_list, gene_name_start_list, gene_name_end_list, gene_len)
                        figures.append(fig)

                except:
                    print('error!!!! in row index', gene)
                    
                    
            if len(figures)>0:                    
                print(f'saving to {fig_dir}patient_{patient_num}/{bed_name[:-4]}_breakpoints.pdf')
                save_figures_to_pdf(f'{fig_dir}patient_{patient_num}/{bed_name[:-4]}_plot.pdf', figures)
            plt.close('all')

                            
    except:
        print('error in file', fa_name)


In [31]:
fig_dir='/pollard/data/projects/shzhang/mdanderson/figs/20241119_bypatient_genecenter_change/'

#run for all patients
for patient_num in range(1,25):
#for patient_num in range(1,2):
    print('running patient ', patient_num)
    dir_patient=f'{patient_data_dir}patient_{patient_num}/walks_fasta/'
    dir_files=os.listdir(dir_patient)
    
    #run for all walks in patient 
    for f in range (0, len(dir_files)):
    #for f in range (0,5):

        print('running file ', dir_files[f])
        pred_akita_genecenter_COSMIC(fa_dir=dir_patient, fa_name=dir_files[f], patient_num=patient_num)

running patient  1
running file  226296-WG01_walk_5_MYCL_containing.fa
bed file     chr     start        end strand  cn     5      span  span_cumsum  bed_idx
0  chr1         1   17349689      +   4     1  17349688     17349688        0
1  chr1  17349690   17351158      +   4     2      1468     17351156        1
2  chr1  17351159   53225853      +   5     3  35874694     53225850        2
3  chr3  98322576   99224962      +   5  2220    902386     54128236        3
4  chr3  99224963   99230671      +   4  2221      5708     54133944        4
5  chr3  99230672  109150007      +   5  2222   9919335     64053279        5
gene TNFRSF14
gene SKI
gene RPL22
gene CASP9
gene MTOR
gene PRDM2
gene SPEN
gene SDHB
gene CAMTA1
gene PRDM16
gene ID3
gene MYCL
gene PAX7
gene LCK
gene SFPQ
gene THRAP3
gene CSF3R
gene MPL
gene MUTYH
gene STIL
gene TAL1
gene CDKN2C
gene ARHGEF10L
gene MDS2
gene ARID1A
gene EPS15
gene TFG
gene CBLB
saving to /pollard/data/projects/shzhang/mdanderson/figs/20241119_bypatien

/tmp/ipykernel_1359333/1934763841.py:8: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  fig=plt.figure(figsize=(7.5,6))


gene AMER1
gene MSN
gene MED12
gene NONO
gene ZMYM3
gene BTK
gene ELF4
gene PHF6
gene TMSB4X
gene USP9X
gene KDM5C
gene AR
gene SEPT6
gene STAG2
gene GPC3
gene KDM6A
gene RBM10
gene ATRX
gene BCORL1
gene CRLF2
gene P2RY8
saving to /pollard/data/projects/shzhang/mdanderson/figs/20241119_bypatient_genecenter_change/patient_1/226298-WG01_walk_1_MYCL_containing_breakpoints.pdf
running file  226296-WG01_walk_4_MYCL_containing.fa
bed file     chr      start        end strand  cn  5      span  span_cumsum  bed_idx
0  chr1  150228403  150481802      -  12 -8    253399       253399        0
1  chr1  124686403  150228402      -   7 -7  25541999     25795398        1
2  chr1   94954905  124686402      -   4 -6  29731497     55526895        2
3  chr1   94947817   94954904      -   3 -5      7087     55533982        3
4  chr1   53225854   94947816      -   4 -4  41721962     97255944        4
5  chr1   17351159   53225853      -   5 -3  35874694    133130638        5
6  chr1   17349690   17351158  

running file  226298-WG01_walk_1_MYCN_containing.fa
bed file     chr     start       end strand  cn    5      span  span_cumsum  bed_idx
0  chr2         1  42119119      +   2  831  42119118     42119118        0
1  chr2  42119120  44249371      +   3  832   2130251     44249369        1
gene MYCN
gene WDCP
gene ASXL2
gene NCOA1
gene DNMT3A
gene STRN
gene ALK
gene BIRC6
gene EML4
saving to /pollard/data/projects/shzhang/mdanderson/figs/20241119_bypatient_genecenter_change/patient_1/226298-WG01_walk_1_MYCN_containing_breakpoints.pdf
running file  226295-WG01_walk_1_MYCN_containing.fa
bed file     chr     start       end strand  cn     5      span  span_cumsum  bed_idx
0  chr2         1  87433202      +   2  1241  87433201     87433201        0
1  chr2  87433203  88729728      +   3  1242   1296525     88729726        1
2  chr2  88732766  90381211      +   3  1244   1648445     90378171        2
3  chr2  90381212  90393319      +   2  1245     12107     90390278        3
4  chr2  9039332

gene NCOA2
gene PREX2
gene CNBD1
gene RSPO2
gene CSMD3
gene FAM135B
saving to /pollard/data/projects/shzhang/mdanderson/figs/20241119_bypatient_genecenter_change/patient_1/226295-WG01_walk_1_MYC_containing_breakpoints.pdf
running file  226297-WG01_walk_1_MYC_containing.fa
bed file     chr      start        end strand  cn    5      span  span_cumsum  bed_idx
0  chr8          1   42520419      +   2  112  42520418     42520418        0
1  chr8   42522790  131265479      +   2  114  88742689    131263107        1
2  chr8  131268277  145138636      +   2  116  13870359    145133466        2
gene LEPROTL1
gene FGFR1
gene IKBKB
gene ARHGEF10
gene PCM1
gene WRN
gene NRG1
gene NSD3
gene ANK1
gene KAT6A
gene HEY1
gene MYC
gene CHCHD7
gene PLAG1
gene NBN
gene CDH17
gene COX6C
gene PABPC1
gene EIF3E
gene RAD21
gene TCEA1
gene LYN
gene RUNX1T1
gene UBR5
gene EXT1
gene HOOK3
gene NCOA2
gene PREX2
gene CNBD1
gene RSPO2
gene CSMD3
gene RECQL4
gene NDRG1
gene FAM135B
saving to /pollard/data/projects/s

...

running file  227183-WG01_walk_1_MDM2_containing.fa
bed file       chr      start        end strand  cn    5      span  span_cumsum  \
0    chr3   39053203   50298002      -   4 -160  11244799     11244799   
1    chr3   34490360   39053202      -   2 -159   4562842     15807641   
2    chr3   20779952   34490359      -   3 -158  13710407     29518048   
3    chr3   17641054   20779951      -   2 -157   3138897     32656945   
4    chr3   16479415   17641053      -   4 -156   1161638     33818583   
5   chr12  109754003  110070542      -   3  -54    316539     34135122   
6   chr12  109466603  109754002      -   4  -53    287399     34422521   
7   chr12  109248603  109466602      -   2  -52    217999     34640520   
8   chr12  108699203  109248602      -   3  -51    549399     35189919   
9   chr12  108098603  108699202      -   2  -50    600599     35790518   
10  chr12  107830403  108098602      -   3  -49    268199     36058717   
11  chr12  103569003  107830402      -   2  -48   4

gene MDS2
gene ARID1A
gene JAK1
gene EPS15
gene CASP9
gene PRDM2
gene SPEN
gene SDHB
gene MTOR
gene TNFRSF14
gene SKI
gene RPL22
gene CAMTA1
gene PRDM16
saving to /pollard/data/projects/shzhang/mdanderson/figs/20241119_bypatient_genecenter_change/patient_24/227183-WG01_walk_3_MYCL_containing_breakpoints.pdf
running file  227184-WG01_walk_2_MYCN_containing.fa
bed file      chr     start        end strand  cn    5      span  span_cumsum  bed_idx
0   chr2  99488570  146952476      -   2 -167  47463906     47463906        0
1   chr2  94159603   99487251      -   3 -165   5327648     52791554        1
2   chr2  92224803   94159602      -   6 -164   1934799     54726353        2
3   chr2  65648403   92224802      -   2 -163  26576399     81302752        3
4   chr2  64844603   65648402      -   3 -162    803799     82106551        4
5   chr2  62430403   64844602      -   2 -161   2414199     84520750        5
6   chr2  60673403   62430402      -   3 -160   1756999     86277749        6
7   ch

gene MDS2
gene ARID1A
gene LCK
gene MYCL
gene SFPQ
gene THRAP3
gene CSF3R
gene MPL
gene MUTYH
gene STIL
gene TAL1
gene CDKN2C
gene EPS15
gene JUN
gene BCL10
gene RPL5
gene NRAS
gene FUBP1
gene RBM15
gene TENT5C
gene JAK1
gene TRIM33
gene NOTCH2
gene BCL9
gene ATP1A1
gene PDE4DIP
gene SETDB1
gene MLLT11
gene ARNT
gene S100A7
gene TPM3
gene LMNA
gene NTRK1
gene PRCC
gene FCRL4
gene MUC1
gene SDHC
gene SDHC
gene FCGR2B
gene PRRX1
gene TPR
gene DDR2
gene ABL2
gene PBX1
gene CDC73
gene PTPRC
gene ELF3
saving to /pollard/data/projects/shzhang/mdanderson/figs/20241119_bypatient_genecenter_change/patient_24/227184-WG01_walk_1_MYCL_containing_breakpoints.pdf
running file  227183-WG01_walk_2_MYCN_containing.fa
bed file     chr      start        end strand  cn    5       span  span_cumsum  bed_idx
0  chr2  110080204  242193529      -   2 -133  132113325    132113325        0
1  chr2   90168003  110080203      -   3 -132   19912200    152025525        1
2  chr2     667003   90168002      -   2 -13

gene WIF1
gene HMGA2
gene MDM2
gene PTPRB
gene LRIG3
gene HOXC13
gene HOXC11
gene NAB2
gene GLI1
gene DDIT3
gene CDK4
gene PRPF40B
gene SMARCD1
gene ATF1
gene ACVR1B
gene NACA
gene STAT6
gene KMT2D
gene ERBB3
gene COL2A1
gene ARID2
gene CDKN1B
gene KRAS
gene ETV6
gene ETNK1
gene PPFIBP1
gene PTPN6
gene CHD4
gene ZNF384
gene CCND2
gene ERC1
gene ERC1
gene KDM5A
error!!!! in row index 32
saving to /pollard/data/projects/shzhang/mdanderson/figs/20241119_bypatient_genecenter_change/patient_24/227184-WG01_walk_1_KRAS_containing_breakpoints.pdf
running file  227184-WG01_walk_3_MYC_containing.fa
bed file     chr      start        end strand  cn    5      span  span_cumsum  bed_idx
0  chr7   32384003   32853802      +  13  353    469799       469799        0
1  chr7   32853803   33062602      +  15  354    208799       678598        1
2  chr7   33062603   33619602      +  12  355    556999      1235597        2
3  chr7   33619603   34047894      +  10  356    428291      1663888        3
4  ch

gene SETDB1
gene MLLT11
gene ARNT
gene S100A7
gene TPM3
gene LMNA
gene NTRK1
gene PRCC
gene FCRL4
gene MUC1
gene SDHC
gene SDHC
gene ELF3
gene CDC73
gene PTPRC
gene FCGR2B
gene PRRX1
gene TPR
gene DDR2
gene ABL2
gene PBX1
gene SDHC
gene SDHC
saving to /pollard/data/projects/shzhang/mdanderson/figs/20241119_bypatient_genecenter_change/patient_24/227184-WG01_walk_2_MYCL_containing_breakpoints.pdf
running file  227183-WG01_walk_2_MYCL_containing.fa
bed file       chr      start        end strand  cn   5      span  span_cumsum  bed_idx
0   chr17    7630309   15848059      +   2  99   8217750      8217750        0
1    chr1    1625793   10422445      +   5   2   8796652     17014402        1
2    chr1   10422446   13155202      +   6   3   2732756     19747158        2
3    chr1   13155203   17351158      +   7   4   4195955     23943113        3
4    chr1   17351159   67596602      +   5   5  50245443     74188556        4
5    chr1   67596603  108426002      +   4   6  40829399    1150179

In [ ]:
bed_name=''
bed=pd.read_csv(f'{patient_data_dir}walks_bed/' + bed_name, sep='\t', header=None)
bed

In [ ]:
patient1_files=os.listdir(f'{patient_data_dir}/patient_1/walks_fasta')
patient1_files

In [ ]:
#make akita predictions for each walk and plot 
def pred_akita_bkpts(fa_dir, fa_name, patient_num):
    bed_name=fa_name.replace('fa', 'bed')
    name_parts = bed_name.rsplit('_', 2)

    gencode_coord=gene_annot[gene_annot.Gene == name_parts[1]]    
    gene_len=abs(gencode_coord.end_gene-gencode_coord.start_gene)
    gene_mid_coord=np.floor(gencode_coord.start_gene + (gene_len/2))
    #print(gencode_coord)
    
    try:

        #read in bed file and figure out where to center the file 
        bed=pd.read_csv(f'{patient_data_dir}walks_bed/' + bed_name, sep='\t', header=None)
        bed.columns = ['chr', 'start', 'end', 'strand', 'cn', '5']
        bed['chr'] = bed['chr'].astype('str') 
        bed['span']=abs(bed['end']-bed['start']) 
        bed['span_cumsum'] = bed['span'].cumsum()
    
        
        #print(bed)
        
        
        #get the row where gene_mid_coord 
        start_condition = ((bed['chr']==gencode_coord.CHROM.values[0])
                & (bed['start'] <= int(gencode_coord.start_gene)) & (int(gencode_coord.start_gene) <= bed['end']) ) 
        row_gene_start_list = start_condition.index[start_condition].tolist()
            
        end_condition = ((bed['chr']==gencode_coord.CHROM.values[0])
                & (bed['start'] <= int(gencode_coord.end_gene)) & (int(gencode_coord.end_gene) <= bed['end']) ) 
        row_gene_end_list = end_condition.index[end_condition].tolist()
        #print('row_gene_start_list', row_gene_start_list)
    
    except: 
        print('bed file not found!!!!!!!!')
       

    
    # read in fasta file 
    seq=read_fasta(f'{fa_dir}{fa_name}')
            
    
    figures=[]
    for i in range (0, len(bed)-1):
    #for i in range(0,4):
        #print('row ', i)
        
        try:
        
            #get first, last rows, and subsets of bed before and after bkpt 
            first_row=bed.iloc[0,:]
            last_row=bed.iloc[(len(bed)-1), :]
            left_bkpt=bed.iloc[i, :]
            right_bkpt=bed.iloc[i+1,:]
            bed_to_bkpt=bed.iloc[0:(i+1),:]
        
            #get relevant lengths
            total_span=np.sum(bed['span'])
            len_left_to_bkpt=np.sum(bed_to_bkpt['span'])
            len_bkpt_to_right=total_span-len_left_to_bkpt
        
        
        
            # left sequence 
            if len_left_to_bkpt>=utils.half_patch_size:
                ALT_left=seq[(len_left_to_bkpt - utils.half_patch_size ):len_left_to_bkpt]
            
            else:   
                #need to pad left from reference 
                diff_len=utils.half_patch_size-len_left_to_bkpt
           
                first_row_idx=first_row.start
                ALT_fromref_left=fasta_hg38.fetch(first_row.chr, first_row_idx-diff_len, first_row_idx).upper()
            
                ALT_left=ALT_fromref_left + seq[0:len_left_to_bkpt]
            
        
            # right sequence 
            if len_bkpt_to_right>utils.half_patch_size:
                ALT_right=seq[len_left_to_bkpt: (len_left_to_bkpt + utils.half_patch_size)]
            
            else:
                #need to pad right from reference        
                diff_len=utils.half_patch_size-len_bkpt_to_right
            
                last_row_idx=last_row.end
                ALT_fromref_right=fasta_hg38.fetch(last_row.chr, last_row_idx, last_row_idx + diff_len).upper()

                ALT_right=seq[len_left_to_bkpt:total_span] + ALT_fromref_right
            
            
            ALT_seq = ALT_left + ALT_right

            
            ########################

            #we want to get indices of the breakpoints, if others are in the window 

            #get index for where the akita window starts 
            left_start=len_left_to_bkpt-utils.half_patch_size
            #print(left_start)

            bkpt_pos_from_left_all=bed['span_cumsum'].tolist() - left_start 
            #print(bkpt_pos_from_left_all)

            #subset to those less than akita window size
            bkpt_pos_from_left=bkpt_pos_from_left_all[(bkpt_pos_from_left_all>0) & (bkpt_pos_from_left_all<utils.MB)]
            #print(bkpt_pos_from_left)
            
            #get the bin index
            bkpt_bin_list_all=[utils.get_bin(bkpt) for bkpt in bkpt_pos_from_left]
            bkpt_bin_list = [bkpt for bkpt in bkpt_bin_list_all if 0 < bkpt <= 448]


            
            ########################
            #finally, get the positions of the gene(s)
                       
            gene_start_bin_list=[]
            gene_end_bin_list=[]

            try:
                
                #check the start of the gene 
                for g in row_gene_start_list:
                    len_to_gene_start=gencode_coord.start_gene.values[0] - (bed.iloc[g, :].start) 
                    #print('len to gene start' , len_to_gene_start)

                    if g>0: 
                        len_before_row=bed.iloc[0:g, :]['span_cumsum'][g-1]
                        len_to_gene_start+=len_before_row


                    gene_start_pos_window=len_to_gene_start - left_start 
                    if (gene_start_pos_window>0) & (gene_start_pos_window<utils.MB):
                        gene_start_bin_list.append(utils.get_bin(gene_start_pos_window))
                        
                #check the end of the gene      
                for g in row_gene_end_list:
                    len_to_gene_end=gencode_coord.end_gene.values[0] - (bed.iloc[g, :].start) 
                    #print('len to gene end' , len_to_gene_end)

                    if g>0: 
                        len_before_row=bed.iloc[0:g, :]['span_cumsum'][g-1]
                        len_to_gene_end+=len_before_row

                    gene_end_pos_window=len_to_gene_end - left_start 
                    if (gene_end_pos_window>0) & (gene_end_pos_window<utils.MB):
                        gene_end_bin_list.append(utils.get_bin(gene_end_pos_window))

                
            except:
                print("error in gene annotation!")


            #########################
            #finally, get the REF -- for the two sequences to the left and right of the breakpoint 
            left_ref_coords=[left_bkpt.chr, left_bkpt.end-utils.half_patch_size, 
                                          left_bkpt.end+utils.half_patch_size]
            right_ref_coords=[right_bkpt.chr, right_bkpt.start-utils.half_patch_size, 
                                          right_bkpt.start+utils.half_patch_size]


            REF_seq_left=fasta_hg38.fetch(left_ref_coords[0], left_ref_coords[1], left_ref_coords[2]).upper()
            REF_seq_right=fasta_hg38.fetch(right_ref_coords[0], right_ref_coords[1], right_ref_coords[2]).upper()


            ####################

            #check sequence lengths, and then run akita 
            #print(len(REF_seq_left))
            assert len(REF_seq_left) == utils.seq_length, 'REF_seq_left is not right length'
            assert len(REF_seq_right) == utils.seq_length, 'REF_seq_right is not right length'
            assert len(ALT_seq) == utils.seq_length, 'ALT_seq is not right length'



            #this is where akita is actually run, output vector size is (99681,)
            REF_left_vec, REF_right_vec, ALT_vec = utils.vector_from_seq(REF_seq_left), utils.vector_from_seq(REF_seq_right), utils.vector_from_seq(ALT_seq)
            REF_left_mat, REF_right_mat, ALT_mat = utils.mat_from_vector(REF_left_vec), utils.mat_from_vector(REF_right_vec), utils.mat_from_vector(ALT_vec)

    
            #order is REF_left, REF_right, ALT
            mat_list=[REF_left_mat, REF_right_mat, ALT_mat]
            
            #add annotation for span between bkpts if same chr, or annot as translocation
            if left_ref_coords[0]==left_ref_coords[0]:
                span_annot=str(right_ref_coords[1] - left_ref_coords[2]) + ' bp between breakpoints'
            else:
                span_annot='interchromosomal'
            
            #get label list 
            label_list=['{}:{}-{}'.format(left_ref_coords[0], left_ref_coords[1], left_ref_coords[2]), 
                       '{}:{}-{}'.format(right_ref_coords[0], right_ref_coords[1], right_ref_coords[2]), 
                       'Left: {}:{}-{} \n Right: {}:{}-{} \n {}'.format(
                        left_bkpt.chr, left_bkpt.start, left_bkpt.end, 
                           right_bkpt.chr, right_bkpt.start, right_bkpt.end, 
                           span_annot), f'{fa_name[:-3]} \n rows {i} - {i+1}']

            #print(label_list)
            # plot!
            fig=plot_complex_pred(mat_list, label_list, bkpt_bin_list, gene_start_bin_list, gene_end_bin_list)
            figures.append(fig)
           

        
        except:
            print('error!!!! in row index', i )
            
    #print(f'saving to {fig_dir}patient_{patient_num}/{bed_name[:-4]}_breakpoints.pdf')
    save_figures_to_pdf(f'{fig_dir}patient_{patient_num}/{bed_name[:-4]}_plot.pdf', figures)
    plt.close('all')
